In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_openml
from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    StratifiedKFold,
    cross_val_predict
)
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

In [ ]:
print("Loading MNIST...")

mnist = fetch_openml(
    'mnist_784',
    version=1,
    as_frame=False
)

X = mnist.data.astype(np.float32)
y = mnist.target.astype(int)

print("Loading Compeleted")

In [ ]:
X

In [ ]:
y

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [ ]:
print(X_train.shape)
print(X_test.shape)
# (Samples , Features)
# (Rows , Columns)

In [ ]:
# StandardScaler normalizes the data so that all features have a similar scale, helping the machine learning model train more effectively.
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
X_test_scaled

In [ ]:
# param_grid = {
#     'C': [0.1, 1, 10],
#     'solver': ['liblinear'],
#     'max_iter': [500]
# }
param_grid = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100],
    'solver': ['lbfgs'],
    'max_iter': [1000],
    'penalty': ['l2']
}

In [ ]:
n_classes = 10

meta_train = np.zeros(
    (X_train_scaled.shape[0], n_classes)
)

meta_test = np.zeros(
    (X_test_scaled.shape[0], n_classes)
)

best_models = {}
best_params = {}

In [ ]:
# 6 combinations × 3 folds = 18 trainings
# 18 × 10 = 180 trainings

In [ ]:
outer_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

for digit in range(10):

    print("\n" + "="*60)
    print(f"Training model for Digit {digit} vs Rest")
    print("="*60)

    y_binary = (y_train == digit).astype(int)
    # Converts the multiclass problem into a binary problem (target digit = 1, others = 0).

    # Hyperparameter tuning
    grid = GridSearchCV(
        LogisticRegression(),
        param_grid,
        cv=3,
        scoring='accuracy',
        n_jobs=3
    )


    grid.fit(X_train_scaled, y_binary)

    best_model = grid.best_estimator_

    best_models[digit] = best_model
    best_params[digit] = grid.best_params_

    print("Best Params:", grid.best_params_)

     
    # OOF Predictions (for stacking)
    # OOF predictions are predictions made on data that the model has not seen during training.
    oof_probs = cross_val_predict(
        best_model,
        X_train_scaled,
        y_binary,
        cv=outer_cv,
        method="predict_proba",
        n_jobs=3
    )[:, 1]

    # So oof_probs is a 1D NumPy array containing the probability that each sample belongs to the target digit (class 1).
    meta_train[:, digit] = oof_probs

    # Train final model
    best_model.fit(X_train_scaled, y_binary)

    # Binary accuracy
    y_train_pred = best_model.predict(X_train_scaled)
    # The trained model predicts whether each image is the target digit or not.
    # [1, 0, 1, 0]
    
    binary_acc = accuracy_score(y_binary, y_train_pred)
    # Compares actual labels with predicted labels and calculates the percentage of correct predictions.


    print("Binary Accuracy:", binary_acc)
    # Test probabilities
    test_probs = best_model.predict_proba(X_test_scaled)[:, 1]
    # Calculates the probability that each test image belongs to the target digit
    # [0.02, 0.95, 0.10]

    meta_test[:, digit] = test_probs
    # Stores these probabilities in the corresponding digit column of meta_test for use by the meta-classifier.

In [ ]:
# Each row in meta_train represents an image, and each column stores the probability predicted by one digit model. These probabilities become the input features for the meta-classifier


In [ ]:
print("Meta train shape:", meta_train.shape)
print("Meta test shape:", meta_test.shape)

# (rows, columns)
# (samples, features)


In [ ]:
meta_train

In [ ]:
meta_model = LogisticRegression(
    
    max_iter=1000
)

meta_model.fit(meta_train, y_train)
joblib.dump(meta_model, 'my_model.joblib')

In [ ]:
y_pred = meta_model.predict(meta_test)

In [ ]:
y_pred

In [ ]:
print("Accuracy:", accuracy_score(y_test, y_pred))

In [ ]:
print(
    classification_report(
        y_test,
        y_pred
    )
)

In [ ]:
# Precision = "When the model says YES, how often is it right?"
# Recall = "How many real YES cases did the model find?"
# F1-score = "Overall balance of Precision and Recall."
# Support = "How many actual examples exist."

In [ ]:
cm = confusion_matrix(y_test, y_pred)
# A table that shows how many predictions were correct and incorrect for each class.

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
# Heatmap is a color-coded chart that makes it easier to visualize patterns, trends, and values in data. In classification tasks, it is commonly used to visualize a confusion matrix.

plt.title("Stacked Logistic Regression - MNIST")
plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.show()

In [ ]:
# Actual 0 → Predicted 0 = 1344

In [ ]:
for i in range(10):
    print(f"Image {i} → Predicted Digit: {y_pred[i]}")

In [ ]:
import matplotlib.pyplot as plt

for i in range(5):
    plt.imshow(X_test[i].reshape(28,28), cmap='gray')
    plt.title(f"Predicted: {y_pred[i]}")
    plt.axis("off")
    plt.show()

In [ ]:
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt

def preprocess_image(image_path):
    """
    Convert image to MNIST format:
    - grayscale
    - detect white background
    - invert if needed
    - resize to 28x28
    - scale using fitted scaler
    """

    img = Image.open(image_path).convert("L")

    # Resize
    img = img.resize((28, 28))

    img_array = np.array(img)

    # Detect background color
    mean_pixel = np.mean(img_array)

    # If image background is white
    if mean_pixel > 127:
        img_array = 255 - img_array

    # Normalize same way as MNIST
    img_array = img_array.astype(np.float32)

    # Flatten
    img_flat = img_array.reshape(1, -1)

    # Apply trained scaler
    img_scaled = scaler.transform(img_flat)

    return img_scaled, img_array

In [ ]:
def predict_digit(image_path):
    img_scaled, processed_img = preprocess_image(image_path)

    # Create meta features
    meta_features = np.zeros((1, 10))

    for digit in range(10):
        prob = best_models[digit].predict_proba(img_scaled)[0, 1]
        meta_features[0, digit] = prob

    prediction = meta_model.predict(meta_features)[0]

    plt.imshow(processed_img, cmap="gray")
    plt.title(f"Predicted Digit: {prediction}")
    plt.axis("off")
    plt.show()

    return prediction

In [ ]:
digit = predict_digit("/Users/gazifayazwani/Desktop/ML/MNIST/data/3.png")
print("Predicted Digit:", digit)